# LAB05 · CSV contra Parquet (Sesión 3 · Módulo 3)

Este cuaderno acompaña a la segunda mitad del LAB05 (la parte HDFS se hace en la terminal).
Ejecuta las celdas **en orden** con `Mayús+Enter`. Recuerda la regla del cuaderno: no ejecutes nada que no hayas leído.

> **Ruta BASE:** celdas 1–4 (convertir + medir). **Ruta COMPLETA:** además, explica la diferencia con la palabra *columnar*. **RETO:** al final.

In [ ]:
# Celda 1 — el motor SQL del curso (si falta: pip install duckdb)
import duckdb
duckdb.__version__

In [ ]:
# Celda 2 — convertir las ventas a Parquet (tarda unos segundos)
# ¿Poca RAM (<4 GB)? Descomenta el techo de memoria (mismos resultados, más lento):
# duckdb.sql("SET memory_limit='512MB'")
duckdb.sql("COPY (SELECT * FROM '../datasets/ventas.csv') TO '../datasets/ventas.parquet' (FORMAT PARQUET)")
print("ventas.parquet creado")

In [ ]:
# Celda 3 — medir TAMAÑO (también puedes hacerlo en la terminal con ls -lh)
import os
csv = os.path.getsize('../datasets/ventas.csv'); par = os.path.getsize('../datasets/ventas.parquet')
print(f"CSV:     {csv/1048576:.1f} MiB")
print(f"Parquet: {par/1048576:.1f} MiB   →  ratio {csv/par:.1f}x")

In [ ]:
# Celda 4 — medir VELOCIDAD: la misma agregación sobre ambos formatos.
# Ejecuta esta celda DOS veces y quédate con la segunda (la primera paga la caché de disco).
%time duckdb.sql("SELECT categoria, SUM(unidades*precio_unitario) FROM '../datasets/ventas.csv' GROUP BY categoria").show()
%time duckdb.sql("SELECT categoria, SUM(unidades*precio_unitario) FROM '../datasets/ventas.parquet' GROUP BY categoria").show()

**Dos cosas que habrás notado (y son lecciones, no errores):** los últimos decimales *bailan* entre ejecuciones — al agregar en paralelo, el orden de las sumas cambia y con él los residuos de la coma flotante (por eso el dinero en serio se maneja con tipos DECIMAL: sesión 4) — y las categorías salen **en orden distinto** cada vez: un `GROUP BY` sin `ORDER BY` no garantiza orden ninguno; el orden se pide, no se supone. **Y la contraprueba reina:** suma las cinco categorías — debe darte el mismo total que calculaste con awk en el LAB03. Dos motores, un total: así se verifica de verdad.

**Tu tabla comparativa (entregable base):** anota tamaño y tiempo de cada formato y la ratio.

**Ruta completa:** la diferencia tiene *dos* causas. La compresión es una. ¿Cuál es la otra? (Pista: ¿cuántas columnas necesita esta consulta y cuántas hay que leer en cada formato?)

---
**RETO EXTRA:** repite la comparativa con `access.log` convertido a Parquet (cárgalo primero como CSV de un solo campo, p. ej. con `read_csv('datasets/access.log', delim='\\x01', header=false)`). Formula tu hipótesis **antes** de medir: ¿comprimirá mejor o peor que las ventas? ¿Por qué?